# 01 · Exploratory data analysis (Week 2)

**Owner:** QR · **Data:** universe v1 ([research/universe.md](../universe.md)), **in-sample dates only** (19 days × 6 symbols). The holdout is refused by `common.universe`.

Scope: spread and volatility profiles, intraday seasonality, message and trade rates, depth, and the basic properties of the labels (share of zero moves, missing values). **No feature-label relationships here.** Those are in `research/studies/univariate_ic.py`, so this notebook can't shape the hypotheses after the fact.

Inputs are the 1 s grid book files (`data/book/`) and the feature files (`data/features/`). Every statistic uses only rows where `book_valid` is true.

In [ ]:
import json
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

from common.meta import read_metadata
from common.paths import book_path
from common.universe import load_universe
from features.query import scan_features

u = load_universe()
SYMS, DATES = list(u.symbols), list(u.in_sample)
# Fixed categorical order (one color per symbol, never re-ordered).
COLOR = dict(zip(SYMS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
INK, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 1.6, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False, "figure.dpi": 110,
})
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")
pd.set_option("display.width", 200)

def date_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))

def plain_log_y(ax, fmt):
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 1.5, 2, 3, 5, 7)))
    ax.yaxis.set_major_formatter(FuncFormatter(fmt))
    ax.yaxis.set_minor_formatter(NullFormatter())

def end_labels(ax, xs, ys_by_sym):
    """Direct labels at the right end of each line (identity is never color-alone)."""
    for s, ys in ys_by_sym.items():
        ax.annotate(s.replace("-USD", ""), (xs[-1], ys[-1]), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=8, color=INK)

## Load the in-sample book grid

One pass over the 114 book files. Per-second quantities are aggregated to minutes to keep memory small. Per-second summaries are kept where a distribution is needed.

In [ ]:
NS_MIN = 60_000_000_000
minute_rows, second_stats, daily_rows = [], {s: [] for s in SYMS}, []
for s in SYMS:
    for d in DATES:
        p = book_path(s, d)
        meta = read_metadata(str(p)); tick = float(meta["tick_size"]); rs = json.loads(meta["replay_stats"])
        df = pl.read_parquet(p, columns=["ts_ns", "bid_px_1", "ask_px_1", "bid_sz_1", "ask_sz_1", "book_valid",
                                         "n_book_msgs", "n_trades", "buy_volume", "sell_volume"])
        v = df["book_valid"].to_numpy()
        bid, ask = df["bid_px_1"].to_numpy(), df["ask_px_1"].to_numpy()
        mid = (bid + ask) / 2
        lm = np.log(np.where(v, mid, np.nan))
        r2 = np.r_[np.nan, np.diff(lm)] ** 2
        depth_usd = (df["bid_sz_1"].to_numpy() * bid + df["ask_sz_1"].to_numpy() * ask) / 2
        vol_usd = (df["buy_volume"].to_numpy() + df["sell_volume"].to_numpy()) * np.where(np.isfinite(mid), mid, 0)
        sec = pl.DataFrame({
            "minute": (df["ts_ns"].to_numpy() % 86_400_000_000_000) // NS_MIN,
            "valid": v, "spread_bps": np.where(v, (ask - bid) / mid * 1e4, np.nan),
            "spread_ticks": np.where(v, (ask - bid) / tick, np.nan), "r2": r2,
            "depth_usd": np.where(v, depth_usd, np.nan), "msgs": df["n_book_msgs"].to_numpy(),
            "trades": df["n_trades"].to_numpy(), "vol_usd": vol_usd,
        }, nan_to_null=True)  # invalid seconds are null, so sums and medians skip them
        m = sec.group_by("minute").agg(
            pl.col("valid").mean(), pl.col("spread_bps").median(), pl.col("spread_ticks").median(),
            pl.col("r2").sum().alias("rv2"), pl.col("depth_usd").median(), pl.col("msgs").sum(),
            pl.col("trades").sum(), pl.col("vol_usd").sum(),
        ).with_columns(pl.lit(s).alias("symbol"), pl.lit(d).alias("date"))
        minute_rows.append(m)
        sv = sec.filter(pl.col("valid"))
        second_stats[s].append(sv.select("spread_bps", "spread_ticks", "depth_usd", "msgs", "trades"))
        daily_rows.append({"symbol": s, "date": d, "tick": tick, "rv_day_pct": float(np.sqrt(np.nansum(r2)) * 100),
                           "trades": int(df["n_trades"].sum()), "msgs": int(df["n_book_msgs"].sum()),
                           "vol_usd_m": float(np.nansum(vol_usd) / 1e6), "invalid_s": rs["invalid_seconds"],
                           "resyncs": rs["resyncs"]})
minutes = pl.concat(minute_rows)
daily = pd.DataFrame(daily_rows)
sec_all = {s: pl.concat(v) for s, v in second_stats.items()}
print(f"{len(daily)} symbol-days, {minutes.height:,} symbol-minutes")

## Summary by symbol

Medians over valid seconds (spread, depth) or over days (activity, realized volatility). Daily realized volatility is √Σ r² of 1 s mid log returns over the day, not annualized.

In [ ]:
rows = []
for s in SYMS:
    x, dd = sec_all[s], daily[daily.symbol == s]
    rows.append({
        "symbol": s, "tick": dd.tick.mode()[0], "tick values": dd.tick.nunique(),
        "spread bps (p50)": x["spread_bps"].median(), "spread bps (p90)": x["spread_bps"].quantile(0.9),
        "spread ticks (p50)": x["spread_ticks"].median(),
        "1-tick share": float((x["spread_ticks"] - 1).abs().lt(1e-9).mean()),
        "L1 depth $ (p50)": x["depth_usd"].median(), "trades/day (p50)": dd.trades.median(),
        "book msgs/day (p50)": dd.msgs.median(), "volume $M/day (p50)": dd.vol_usd_m.median(),
        "daily RV % (p50)": dd.rv_day_pct.median(),
    })
summary = pd.DataFrame(rows).set_index("symbol")
summary

## Spread profile

The CDF of the quoted spread in bps (log scale). The table gives the share of valid seconds at each spread width in ticks.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for s in SYMS:
    x = np.sort(sec_all[s]["spread_bps"].drop_nulls().drop_nans().to_numpy())
    x = x[x > 0]
    q = np.linspace(0, 1, len(x))
    step = max(1, len(x) // 4000)
    ax.plot(x[::step], q[::step], color=COLOR[s], label=s)
ax.set_xscale("log"); ax.set_xlabel("quoted spread (bps, log scale)"); ax.set_ylabel("share of valid seconds ≤ x")
ax.set_title("Quoted spread distribution, in-sample"); ax.legend(loc="lower right", ncols=2)
plt.show()

bins = [0, 1.5, 2.5, 5.5, 10.5, 50.5, np.inf]; labels = ["1", "2", "3-5", "6-10", "11-50", ">50"]
tick_tab = pd.DataFrame({s: pd.cut(sec_all[s]["spread_ticks"].to_pandas(), bins, labels=labels).value_counts(normalize=True).reindex(labels)
                         for s in SYMS}).T
tick_tab.columns.name = "spread (ticks)"
tick_tab

## Volatility profile

Daily realized volatility across the in-sample dates (log scale). The holdout months aren't shown.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in SYMS:
    dd = daily[daily.symbol == s]
    ax.plot(pd.to_datetime(dd.date), dd.rv_day_pct, color=COLOR[s], marker="o", markersize=4, label=s)
    end_labels(ax, list(pd.to_datetime(dd.date)), {s: list(dd.rv_day_pct)})
ax.set_yscale("log"); plain_log_y(ax, lambda v, _: f"{v:g}%"); date_axis(ax)
ax.set_ylabel("daily realized vol of mid (log scale)")
ax.set_title("Realized volatility by sample day"); ax.legend(ncols=3, loc="upper left")
plt.show()
daily.pivot(index="date", columns="symbol", values="rv_day_pct")[SYMS].describe().loc[["min", "50%", "max"]]

## Intraday seasonality

Each metric by UTC hour, divided by the symbol's own all-day mean so that all six share one scale (1 = average). The shaded band is the US equity cash session at its summer hours, 13:30–20:00 UTC (09:30–16:00 New York time; in winter it is one hour later, 14:30–21:00 UTC).

In [ ]:
hourly = (minutes.with_columns((pl.col("minute") // 60).alias("hour"))
          .group_by("symbol", "hour")
          .agg(pl.col("trades").mean(), pl.col("msgs").mean(), pl.col("spread_bps").median(),
               pl.col("rv2").mean(), pl.col("vol_usd").mean())
          .sort("symbol", "hour").to_pandas())
metrics = [("trades", "trades per minute"), ("msgs", "book messages per minute"),
           ("spread_bps", "median spread (bps)"), ("rv2", "realized variance per minute")]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (col, title) in zip(axes.flat, metrics):
    for s in SYMS:
        h = hourly[hourly.symbol == s]
        y = h[col] / h[col].mean()
        ax.plot(h.hour, y, color=COLOR[s], label=s)
    ax.axvspan(13.5, 20, color=GRID, alpha=0.5, lw=0)
    ax.axhline(1, color=AXIS, lw=0.8)
    ax.set_title(f"{title}, relative to daily mean"); ax.set_xticks(range(0, 24, 3))
axes[1, 0].set_xlabel("hour (UTC)"); axes[1, 1].set_xlabel("hour (UTC)")
axes[0, 0].legend(ncols=3, fontsize=8, loc="upper left")
fig.tight_layout(); plt.show()
hourly.pivot(index="hour", columns="symbol", values="trades")[SYMS].div(
    hourly.groupby("symbol").trades.mean()[SYMS]).round(2).T

## Message and trade rates

Percentiles of per-second counts over valid seconds.

In [ ]:
pct = [0.5, 0.9, 0.99]
rate = pd.concat({s: pd.DataFrame({
    **{f"msgs/s p{int(q*100)}": [sec_all[s]["msgs"].quantile(q)] for q in pct},
    **{f"trades/s p{int(q*100)}": [sec_all[s]["trades"].quantile(q)] for q in pct},
    "seconds with ≥1 trade": [float((sec_all[s]["trades"] > 0).mean())],
    "seconds with ≥1 book msg": [float((sec_all[s]["msgs"] > 0).mean())],
}) for s in SYMS}).droplevel(1)
rate

## Activity across the sample

Trades per day on each sample date (log scale). This shows how much liquidity moved over the two years, which matters for H3 (buckets are frozen from in-sample medians) and H5.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for s in SYMS:
    dd = daily[daily.symbol == s]
    ax.plot(pd.to_datetime(dd.date), dd.trades, color=COLOR[s], marker="o", markersize=4, label=s)
ax.set_yscale("log"); plain_log_y(ax, lambda v, _: f"{v / 1e3:g}k"); date_axis(ax)
ax.set_ylabel("trades per day (log scale)"); ax.set_title("Trading activity by sample day")
ax.legend(ncols=3, loc="lower left", fontsize=8)
plt.show()
daily.pivot(index="date", columns="symbol", values="trades")[SYMS]

## Labels: zero moves and missing values

Share of valid rows where the mid did not move over the horizon (`y = 0`, which is dropped from the direction labels) and share of rows with a NaN label (book invalid in [t, t+h] or stale mid). The spread of non-zero returns is in bps.

In [ ]:
lab = (scan_features(columns=["y_10s", "y_30s", "y_60s"], with_labels=True).select("symbol", "book_valid", "y_10s", "y_30s", "y_60s")
       .filter(pl.col("book_valid")).collect())
out = []
for s in SYMS:
    g = lab.filter(pl.col("symbol") == s)
    row = {"symbol": s}
    for h in ("10s", "30s", "60s"):
        y = g[f"y_{h}"].to_numpy()
        fin = np.isfinite(y)
        row[f"NaN {h}"] = 1 - fin.mean()
        row[f"zero {h}"] = float(np.mean(y[fin] == 0))
        nz = y[fin & (y != 0)] * 1e4
        row[f"|y| p50 bps {h}"] = float(np.median(np.abs(nz))) if nz.size else np.nan
    out.append(row)
pd.DataFrame(out).set_index("symbol")

## L1 depth

Median dollar depth at the best level (mean of the bid and ask sides) and the first and ninth deciles, over valid seconds.

In [ ]:
pd.DataFrame({s: {q: sec_all[s]["depth_usd"].quantile(q) for q in (0.1, 0.5, 0.9)} for s in SYMS}).T.rename(
    columns={0.1: "p10 $", 0.5: "p50 $", 0.9: "p90 $"})